# 法說會資料整合
- 上市法說會 → `twse_ann_df`
- 上櫃法說會 → `otc_ann_df`
- 篩選條件：召開時間 ≥ 13:30，且落在台股財報公佈期 ±10 交易日內
- 每財報錨點 × 每家公司，只保留錨點前後各最近一筆，按公司分組排序

In [ ]:
import pandas as pd
import glob
import os
import exchange_calendars as xcals

## 輔助函數

In [ ]:
def roc_to_date(roc_str):
    """民國年字串 (e.g. '110/03/16') → pd.Timestamp"""
    try:
        parts = str(roc_str).strip().split('/')
        if len(parts) == 3:
            year = int(parts[0]) + 1911
            return pd.Timestamp(f'{year}/{parts[1]}/{parts[2]}')
    except Exception:
        pass
    return pd.NaT


def build_trading_arr(start_year, end_year):
    """取得 XTAI 台灣交易所的交易日陣列（tz-naive）"""
    cal = xcals.get_calendar('XTAI')
    buf_start = pd.Timestamp(f'{start_year - 1}-11-01')
    buf_end   = min(pd.Timestamp(f'{end_year + 1}-06-30'),
                    cal.last_session.tz_localize(None))
    sessions = cal.sessions_in_range(buf_start, buf_end)
    return pd.DatetimeIndex([s.tz_localize(None) for s in sessions])


def get_report_anchors(start_year, end_year):
    """回傳各年度所有財報截止錨點日期（已排序去重）"""
    monthly = [(m, 10) for m in range(1, 13)]
    special = [(3, 15), (3, 31), (4, 1), (5, 15), (8, 14), (11, 14)]
    anchors = []
    for year in range(start_year, end_year + 1):
        for m, d in monthly + special:
            try:
                anchors.append(pd.Timestamp(f'{year}-{m:02d}-{d:02d}'))
            except Exception:
                pass
    return sorted(set(anchors))


def filter_closest_to_anchors(df, anchors, trading_arr, window=10):
    """
    每個財報錨點 × 每家公司，只保留：
      - 錨點當日（含）之前最近的一筆
      - 錨點之後最近的一筆
    最終按公司代號、日期排序。
    """
    kept = []
    for anchor in anchors:
        idx = trading_arr.searchsorted(anchor, side='right') - 1
        if not (0 <= idx < len(trading_arr)):
            continue
        lo = trading_arr[max(0, idx - window)]
        hi = trading_arr[min(len(trading_arr) - 1, idx + window)]

        win = df[(df['日期'] >= lo) & (df['日期'] <= hi)]
        if win.empty:
            continue

        before = (win[win['日期'] <= anchor]
                  .sort_values('日期')
                  .groupby('公司代號', sort=False)
                  .last()
                  .reset_index())
        after = (win[win['日期'] > anchor]
                 .sort_values('日期')
                 .groupby('公司代號', sort=False)
                 .first()
                 .reset_index())
        kept.extend([before, after])

    if not kept:
        return df.iloc[0:0].copy()

    return (pd.concat(kept, ignore_index=True)
            .drop_duplicates()
            .sort_values(['公司代號', '日期'])
            .reset_index(drop=True))

## 台股財報公佈期間設定

錨點日期（rule.md）：每月 10 日、3/15、3/31、4/1、5/15、8/14、11/14

In [ ]:
trading_arr = build_trading_arr(2021, 2026)
anchors     = get_report_anchors(2021, 2026)
print(f'交易日陣列：{len(trading_arr)} 個交易日')
print(f'財報錨點：{len(anchors)} 個')

## 上市法說會（TWSE）→ twse_ann_df

In [ ]:
twse_path = r'E:\法說會+主動型\上市法說會'
twse_files = sorted(glob.glob(os.path.join(twse_path, '*.csv')))
print(f'找到 {len(twse_files)} 個上市法說會檔案：')
for f in twse_files:
    print(' ', os.path.basename(f))

In [ ]:
twse_dfs = []
for f in twse_files:
    df = pd.read_csv(f, encoding='cp950', encoding_errors='ignore', on_bad_lines='skip')
    twse_dfs.append(df)

twse_ann_df = pd.concat(twse_dfs, ignore_index=True)

# 民國年轉西元日期
twse_ann_df['日期'] = twse_ann_df['召開法人說明會日期'].apply(roc_to_date)

# 只保留 13:30（含）之後的場次
twse_ann_df = twse_ann_df[twse_ann_df['召開法人說明會時間'].astype(str).str.strip() >= '13:30']

# 每財報錨點 × 每公司：只保留錨點前後各最近一筆，按公司分組排序
twse_ann_df = filter_closest_to_anchors(twse_ann_df, anchors, trading_arr, window=20)

print(f'上市法說會共 {len(twse_ann_df):,} 筆（每公司每財報期保留前後各最近一筆）')
twse_ann_df.head(10)

## 上櫃法說會（OTC）→ otc_ann_df

In [ ]:
otc_path = r'E:\法說會+主動型\上櫃法說會'
otc_files = sorted(glob.glob(os.path.join(otc_path, '*.csv')))
print(f'找到 {len(otc_files)} 個上櫃法說會檔案：')
for f in otc_files:
    print(' ', os.path.basename(f))

In [ ]:
otc_dfs = []
for f in otc_files:
    df = pd.read_csv(f, encoding='cp950', encoding_errors='ignore', on_bad_lines='skip')
    otc_dfs.append(df)

otc_ann_df = pd.concat(otc_dfs, ignore_index=True)

# 民國年轉西元日期
otc_ann_df['日期'] = otc_ann_df['召開法人說明會日期'].apply(roc_to_date)

# 只保留 13:30（含）之後的場次
otc_ann_df = otc_ann_df[otc_ann_df['召開法人說明會時間'].astype(str).str.strip() >= '13:30']

# 每財報錨點 × 每公司：只保留錨點前後各最近一筆，按公司分組排序
otc_ann_df = filter_closest_to_anchors(otc_ann_df, anchors, trading_arr, window=20)

print(f'上櫃法說會共 {len(otc_ann_df):,} 筆（每公司每財報期保留前後各最近一筆）')
otc_ann_df.head(10)

## 資料概覽

In [ ]:
print('=== 上市法說會 twse_ann_df ===')
print(twse_ann_df.info())
print()
print('=== 上櫃法說會 otc_ann_df ===')
print(otc_ann_df.info())

In [ ]:
print('上市法說會 - 各年度筆數：')
print(twse_ann_df['日期'].dt.year.value_counts().sort_index())
print()
print('上櫃法說會 - 各年度筆數：')
print(otc_ann_df['日期'].dt.year.value_counts().sort_index())

## 敘述統計：法說會距財報截止日天數

In [ ]:
_anchor_label_map = {
    (3, 15): '年報(大型股/金融)',
    (3, 31): 'Q4財報',
    (4,  1): '年報(其餘)',
    (5, 15): 'Q1財報',
    (8, 14): 'Q2財報',
    (11,14): 'Q3財報',
}

def get_anchor_label(anchor):
    if anchor.day == 10:
        return '月營收'
    return _anchor_label_map.get((anchor.month, anchor.day), "其他")


def attach_anchor_info(df, anchors, trading_arr):
    anchor_series = pd.Series(anchors)

    def _nearest(date):
        if pd.isna(date):
            return (pd.NaT, None)
        diffs = (anchor_series - date).dt.days.abs()
        i = diffs.idxmin()
        a = anchors[i]
        # 交易日差：兩日期在 trading_arr 的索引差
        idx_date   = int(trading_arr.searchsorted(date, side="left"))
        idx_anchor = int(trading_arr.searchsorted(a,    side="left"))
        return (a, idx_date - idx_anchor)

    info = df["日期"].apply(_nearest)
    df = df.copy()
    df["財報錨點"]      = info.apply(lambda x: x[0])
    df["距截止日交易日"] = info.apply(lambda x: x[1])  # 負=截止日前，正=截止日後
    df["財報期別"]      = df["財報錨點"].apply(lambda a: get_anchor_label(a) if pd.notna(a) else None)
    df["財報年度"]      = df["財報錨點"].apply(lambda a: a.year if pd.notna(a) else None)
    df["前後"]         = df["距截止日交易日"].apply(
        lambda x: '截止日後' if x is not None and x > 0 else ('截止日前' if x is not None and x < 0 else '截止日當天')
    )
    return df


twse_stat = attach_anchor_info(twse_ann_df, anchors, trading_arr); twse_stat['市場'] = '上市(TWSE)'
otc_stat  = attach_anchor_info(otc_ann_df,  anchors, trading_arr); otc_stat['市場']  = '上櫃(OTC)'
all_stat  = pd.concat([twse_stat, otc_stat], ignore_index=True)
print(f"合計 {len(all_stat):,} 筆，已附加財報期別與距截止日交易日數")


### 各財報期別 × 年度 × 前後：距截止日天數敘述統計
- 負值 = 截止日前（提前開說明會）
- 正值 = 截止日後（滯後開說明會）

In [ ]:
period_order = ["月營收", "年報(大型股/金融)", "年報(其餘)", "Q1財報", "Q2財報", "Q3財報", "Q4財報", "其他"]

desc = (
    all_stat
    .assign(財報期別=pd.Categorical(all_stat["財報期別"], categories=period_order, ordered=True))
    .groupby(["財報期別", "財報年度", "前後"])["距截止日交易日"]
    .describe()
    .rename(columns={"count":"筆數","mean":"平均","std":"標準差",
                     "min":"最小","25%":"Q1","50%":"中位數","75%":"Q3","max":"最大"})
    .round(1)
)

pd.set_option("display.max_rows", 200)
display(desc)


In [ ]:
# 簡化版：只看季報/年報（排除月營收）
main_periods = ["年報(大型股/金融)", "年報(其餘)", "Q1財報", "Q2財報", "Q3財報", "Q4財報"]
summary = (
    all_stat[all_stat["財報期別"].isin(main_periods)]
    .groupby(["財報期別", "財報年度", "前後"])["距截止日交易日"]
    .agg(筆數="count", 平均交易日="mean", 中位數="median", 最小="min", 最大="max")
    .round(1)
    .reset_index()
)
display(summary)


In [ ]:
q_periods = ["Q1財報", "Q2財報", "Q3財報"]
median_tbl = (
    all_stat[all_stat["財報期別"].isin(q_periods)]
    .groupby(["財報期別", "財報年度", "前後"])["距截止日交易日"]
    .median()
    .rename("中位數(交易日)")
    .reset_index()
    .pivot_table(index=["財報期別", "財報年度"], columns="前後", values="中位數(交易日)")
    .rename_axis(columns=None)
    [["截止日前", "截止日後"]]
    .round(1)
)
display(median_tbl)


In [ ]:
sample = (
    all_stat[
        (all_stat["財報期別"] == "Q1財報") &
        (all_stat["財報年度"] == 2025)
    ]
    .sort_values("距截止日交易日")
    .reset_index(drop=True)
)
print(f"Q1財報 2025 共 {len(sample)} 筆")
display(sample)


## 輸出互動查詢網頁 → 法說會查詢.html

In [ ]:
import json as _json

# 準備輸出欄位
_export_cols = ['公司代號', '日期', '召開法人說明會時間',
                '財報期別', '財報年度', '財報錨點', '前後', '距截止日交易日', '市場']
if '公司名稱' in all_stat.columns:
    _export_cols.insert(1, '公司名稱')
_export_cols = [c for c in _export_cols if c in all_stat.columns]

_ex = all_stat[_export_cols].copy()
_ex['日期'] = _ex['日期'].dt.strftime('%Y-%m-%d')
_ex['財報錨點'] = _ex['財報錨點'].apply(
    lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else None)
_data_json = _json.dumps(_ex.to_dict(orient='records'), ensure_ascii=False)

_template = '<!DOCTYPE html>\n<html lang="zh-TW">\n<head>\n<meta charset="UTF-8">\n<title>法說會財報對應查詢</title>\n<style>\n*{box-sizing:border-box;margin:0;padding:0;}\nbody{font-family:\'Segoe UI\',Tahoma,sans-serif;display:flex;height:100vh;overflow:hidden;background:#f1f5f9;}\n#sb{width:230px;min-width:230px;background:#1e293b;display:flex;flex-direction:column;height:100vh;}\n#sb-title{padding:18px 16px 6px;color:#60a5fa;font-size:13px;font-weight:700;letter-spacing:.5px;}\n#sb-sub{padding:0 16px 10px;color:#64748b;font-size:11px;}\n#srch{margin:4px 12px 8px;padding:8px 10px;border:none;border-radius:6px;background:#334155;color:#e2e8f0;font-size:13px;outline:none;width:calc(100% - 24px);}\n#srch::placeholder{color:#4b5563;}\n#clist{flex:1;overflow-y:auto;}\n.ci{padding:8px 14px 8px 16px;color:#cbd5e1;font-size:13px;cursor:pointer;border-left:3px solid transparent;display:flex;flex-direction:column;}\n.ci:hover{background:#334155;}\n.ci.active{background:#1e3a5f;border-left-color:#60a5fa;color:#fff;}\n.cc{font-weight:600;}\n.cn{font-size:11px;color:#94a3b8;margin-top:2px;}\n#main{flex:1;overflow-y:auto;padding:28px 32px;}\n#ph{color:#94a3b8;text-align:center;margin-top:100px;font-size:15px;}\n#detail{display:none;}\n#ch{font-size:22px;font-weight:700;color:#1e293b;margin-bottom:20px;}\n#ch small{color:#64748b;font-weight:400;font-size:15px;margin-left:10px;}\n#ybtns{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:28px;}\n.yb{padding:6px 20px;border-radius:20px;border:1.5px solid #cbd5e1;background:#fff;color:#475569;font-size:13px;cursor:pointer;font-weight:500;transition:.15s;}\n.yb:hover{border-color:#93c5fd;color:#1d4ed8;}\n.yb.active{background:#3b82f6;border-color:#3b82f6;color:#fff;}\n#qs{display:flex;flex-direction:column;gap:16px;}\n.qc{background:#fff;border-radius:10px;box-shadow:0 1px 4px rgba(0,0,0,.08);overflow:hidden;}\n.qh{padding:12px 18px;background:#f8fafc;font-weight:700;font-size:14px;color:#1e293b;border-bottom:1px solid #e2e8f0;}\n.qbody{padding:16px 18px;display:flex;gap:28px;flex-wrap:wrap;}\n.qcol{flex:1;min-width:200px;}\n.qst{font-size:11px;font-weight:700;letter-spacing:.5px;text-transform:uppercase;margin-bottom:8px;padding:3px 10px;border-radius:4px;display:inline-block;}\n.st-b{background:#fef3c7;color:#92400e;}\n.st-a{background:#dcfce7;color:#166534;}\n.st-s{background:#dbeafe;color:#1e40af;}\ntable{width:100%;border-collapse:collapse;font-size:12.5px;}\nth{text-align:left;padding:4px 6px;color:#94a3b8;font-size:11px;font-weight:600;}\ntd{padding:5px 6px;color:#374151;border-top:1px solid #f1f5f9;}\n.dn{color:#b45309;font-weight:700;}\n.dp{color:#15803d;font-weight:700;}\n.dz{color:#1d4ed8;font-weight:700;}\n.empty{color:#94a3b8;font-size:12px;font-style:italic;padding:4px 0;}\n</style>\n</head>\n<body>\n<div id="sb">\n  <div id="sb-title">法說會財報對應查詢</div>\n  <div id="sb-sub" id="cnt"></div>\n  <input id="srch" placeholder="搜尋代號或名稱…" oninput="filterList(this.value)">\n  <div id="clist"></div>\n</div>\n<div id="main">\n  <div id="ph">← 請從左側選擇公司</div>\n  <div id="detail">\n    <div id="ch"></div>\n    <div id="ybtns"></div>\n    <div id="qs"></div>\n  </div>\n</div>\n<script>\nconst RECORDS = __DATA__;\n\nconst companies = {};\nRECORDS.forEach(r => {\n  const code = String(r[\'公司代號\'] || \'\').trim();\n  if (!code) return;\n  if (!companies[code]) companies[code] = { name: r[\'公司名稱\'] || \'\', data: {} };\n  const yr = String(r[\'財報年度\'] || \'\');\n  if (!yr) return;\n  if (!companies[code].data[yr]) companies[code].data[yr] = {};\n  const period = r[\'財報期別\'] || \'其他\';\n  if (!companies[code].data[yr][period])\n    companies[code].data[yr][period] = { before: [], after: [], same: [] };\n  const slot = r[\'前後\'] === \'截止日前\' ? \'before\' : r[\'前後\'] === \'截止日後\' ? \'after\' : \'same\';\n  companies[code].data[yr][period][slot].push(r);\n});\n\nconst sortedCodes = Object.keys(companies).sort();\ndocument.getElementById(\'sb-sub\').textContent = sortedCodes.length + \' 家公司\';\n\nlet activeCode = null;\n\nfunction renderList(filter) {\n  const f = (filter || \'\').trim();\n  document.getElementById(\'clist\').innerHTML = sortedCodes\n    .filter(c => !f || c.includes(f) || (companies[c].name || \'\').includes(f))\n    .map(c => `<div class="ci${c === activeCode ? \' active\' : \'\'}" onclick="selectCompany(\'${c}\')">\n      <span class="cc">${c}</span>\n      ${companies[c].name ? `<span class="cn">${companies[c].name}</span>` : \'\'}\n    </div>`).join(\'\');\n}\nfunction filterList(v) { renderList(v); }\nrenderList(\'\');\n\nfunction selectCompany(code) {\n  activeCode = code;\n  renderList(document.getElementById(\'srch\').value);\n  document.getElementById(\'ph\').style.display = \'none\';\n  document.getElementById(\'detail\').style.display = \'block\';\n\n  const co = companies[code];\n  document.getElementById(\'ch\').innerHTML =\n    `${code}<small>${co.name || \'\'}</small>`;\n\n  const years = Object.keys(co.data).sort((a, b) => b - a);\n  document.getElementById(\'ybtns\').innerHTML =\n    years.map(y => `<button class="yb" onclick="selectYear(\'${y}\')">${y}</button>`).join(\'\');\n  document.getElementById(\'qs\').innerHTML = \'\';\n\n  if (years.length === 1) selectYear(years[0]);\n}\n\nconst PERIOD_ORDER = [\'Q1財報\',\'Q2財報\',\'Q3財報\',\'Q4財報\',\'年報(大型股/金融)\',\'年報(其餘)\',\'月營收\',\'其他\'];\n\nfunction selectYear(yr) {\n  document.querySelectorAll(\'.yb\').forEach(b =>\n    b.classList.toggle(\'active\', b.textContent === yr));\n\n  const data = companies[activeCode].data[yr] || {};\n  const qs = document.getElementById(\'qs\');\n  qs.innerHTML = \'\';\n\n  PERIOD_ORDER.forEach(period => {\n    if (!data[period]) return;\n    const { before, after, same } = data[period];\n    if (!before.length && !after.length && !same.length) return;\n\n    const sB = [...before].sort((a, b) => a[\'距截止日交易日\'] - b[\'距截止日交易日\']);\n    const sA = [...after].sort((a, b) => a[\'距截止日交易日\'] - b[\'距截止日交易日\']);\n\n    function tbl(arr, cls) {\n      if (!arr.length) return \'<div class="empty">無資料</div>\';\n      return `<table>\n        <tr><th>日期</th><th>時間</th><th>交易日差</th><th>財報截止日</th><th>市場</th></tr>\n        ${arr.map(r => `<tr>\n          <td>${r[\'日期\'] || \'\'}</td>\n          <td>${r[\'召開法人說明會時間\'] || \'\'}</td>\n          <td class="${cls}">${r[\'距截止日交易日\']}</td>\n          <td>${r[\'財報錨點\'] || \'\'}</td>\n          <td>${r[\'市場\'] || \'\'}</td>\n        </tr>`).join(\'\')}\n      </table>`;\n    }\n\n    const div = document.createElement(\'div\');\n    div.className = \'qc\';\n    div.innerHTML = `\n      <div class="qh">${period}</div>\n      <div class="qbody">\n        <div class="qcol">\n          <div class="qst st-b">截止日前（${before.length} 筆）</div>\n          ${tbl(sB, \'dn\')}\n        </div>\n        <div class="qcol">\n          <div class="qst st-a">截止日後（${after.length} 筆）</div>\n          ${tbl(sA, \'dp\')}\n        </div>\n        ${same.length ? `<div class="qcol">\n          <div class="qst st-s">截止日當天（${same.length} 筆）</div>\n          ${tbl(same, \'dz\')}\n        </div>` : \'\'}\n      </div>`;\n    qs.appendChild(div);\n  });\n}\n</script>\n</body>\n</html>'

_html_out = _template.replace('__DATA__', _data_json)
_out_path = r'E:\法說會+主動型\法說會查詢.html'
with open(_out_path, 'w', encoding='utf-8') as _f:
    _f.write(_html_out)
print(f'已輸出：{_out_path}')
print(f'共 {len(_ex):,} 筆資料已嵌入，請用瀏覽器直接開啟 法說會查詢.html')


## 輸出互動查詢網頁 → 法說會查詢.html

In [ ]:
import json as _json

# 準備輸出欄位
_export_cols = ['公司代號', '日期', '召開法人說明會時間',
                '財報期別', '財報年度', '財報錨點', '前後', '距截止日交易日', '市場']
if '公司名稱' in all_stat.columns:
    _export_cols.insert(1, '公司名稱')
_export_cols = [c for c in _export_cols if c in all_stat.columns]

_ex = all_stat[_export_cols].copy()
_ex['日期'] = _ex['日期'].dt.strftime('%Y-%m-%d')
_ex['財報錨點'] = _ex['財報錨點'].apply(
    lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else None)
_data_json = _json.dumps(_ex.to_dict(orient='records'), ensure_ascii=False)

_template = '<!DOCTYPE html>\n<html lang="zh-TW">\n<head>\n<meta charset="UTF-8">\n<title>法說會財報對應查詢</title>\n<style>\n*{box-sizing:border-box;margin:0;padding:0;}\nbody{font-family:\'Segoe UI\',Tahoma,sans-serif;display:flex;height:100vh;overflow:hidden;background:#f1f5f9;}\n#sb{width:230px;min-width:230px;background:#1e293b;display:flex;flex-direction:column;height:100vh;}\n#sb-title{padding:18px 16px 6px;color:#60a5fa;font-size:13px;font-weight:700;letter-spacing:.5px;}\n#sb-sub{padding:0 16px 10px;color:#64748b;font-size:11px;}\n#srch{margin:4px 12px 8px;padding:8px 10px;border:none;border-radius:6px;background:#334155;color:#e2e8f0;font-size:13px;outline:none;width:calc(100% - 24px);}\n#srch::placeholder{color:#4b5563;}\n#clist{flex:1;overflow-y:auto;}\n.ci{padding:8px 14px 8px 16px;color:#cbd5e1;font-size:13px;cursor:pointer;border-left:3px solid transparent;display:flex;flex-direction:column;}\n.ci:hover{background:#334155;}\n.ci.active{background:#1e3a5f;border-left-color:#60a5fa;color:#fff;}\n.cc{font-weight:600;}\n.cn{font-size:11px;color:#94a3b8;margin-top:2px;}\n#main{flex:1;overflow-y:auto;padding:28px 32px;}\n#ph{color:#94a3b8;text-align:center;margin-top:100px;font-size:15px;}\n#detail{display:none;}\n#ch{font-size:22px;font-weight:700;color:#1e293b;margin-bottom:20px;}\n#ch small{color:#64748b;font-weight:400;font-size:15px;margin-left:10px;}\n#ybtns{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:28px;}\n.yb{padding:6px 20px;border-radius:20px;border:1.5px solid #cbd5e1;background:#fff;color:#475569;font-size:13px;cursor:pointer;font-weight:500;transition:.15s;}\n.yb:hover{border-color:#93c5fd;color:#1d4ed8;}\n.yb.active{background:#3b82f6;border-color:#3b82f6;color:#fff;}\n#qs{display:flex;flex-direction:column;gap:16px;}\n.qc{background:#fff;border-radius:10px;box-shadow:0 1px 4px rgba(0,0,0,.08);overflow:hidden;}\n.qh{padding:12px 18px;background:#f8fafc;font-weight:700;font-size:14px;color:#1e293b;border-bottom:1px solid #e2e8f0;}\n.qbody{padding:16px 18px;display:flex;gap:28px;flex-wrap:wrap;}\n.qcol{flex:1;min-width:200px;}\n.qst{font-size:11px;font-weight:700;letter-spacing:.5px;text-transform:uppercase;margin-bottom:8px;padding:3px 10px;border-radius:4px;display:inline-block;}\n.st-b{background:#fef3c7;color:#92400e;}\n.st-a{background:#dcfce7;color:#166534;}\n.st-s{background:#dbeafe;color:#1e40af;}\ntable{width:100%;border-collapse:collapse;font-size:12.5px;}\nth{text-align:left;padding:4px 6px;color:#94a3b8;font-size:11px;font-weight:600;}\ntd{padding:5px 6px;color:#374151;border-top:1px solid #f1f5f9;}\n.dn{color:#b45309;font-weight:700;}\n.dp{color:#15803d;font-weight:700;}\n.dz{color:#1d4ed8;font-weight:700;}\n.empty{color:#94a3b8;font-size:12px;font-style:italic;padding:4px 0;}\n</style>\n</head>\n<body>\n<div id="sb">\n  <div id="sb-title">法說會財報對應查詢</div>\n  <div id="sb-sub" id="cnt"></div>\n  <input id="srch" placeholder="搜尋代號或名稱…" oninput="filterList(this.value)">\n  <div id="clist"></div>\n</div>\n<div id="main">\n  <div id="ph">← 請從左側選擇公司</div>\n  <div id="detail">\n    <div id="ch"></div>\n    <div id="ybtns"></div>\n    <div id="qs"></div>\n  </div>\n</div>\n<script>\nconst RECORDS = __DATA__;\n\nconst companies = {};\nRECORDS.forEach(r => {\n  const code = String(r[\'公司代號\'] || \'\').trim();\n  if (!code) return;\n  if (!companies[code]) companies[code] = { name: r[\'公司名稱\'] || \'\', data: {} };\n  const yr = String(r[\'財報年度\'] || \'\');\n  if (!yr) return;\n  if (!companies[code].data[yr]) companies[code].data[yr] = {};\n  const period = r[\'財報期別\'] || \'其他\';\n  if (!companies[code].data[yr][period])\n    companies[code].data[yr][period] = { before: [], after: [], same: [] };\n  const slot = r[\'前後\'] === \'截止日前\' ? \'before\' : r[\'前後\'] === \'截止日後\' ? \'after\' : \'same\';\n  companies[code].data[yr][period][slot].push(r);\n});\n\nconst sortedCodes = Object.keys(companies).sort();\ndocument.getElementById(\'sb-sub\').textContent = sortedCodes.length + \' 家公司\';\n\nlet activeCode = null;\n\nfunction renderList(filter) {\n  const f = (filter || \'\').trim();\n  document.getElementById(\'clist\').innerHTML = sortedCodes\n    .filter(c => !f || c.includes(f) || (companies[c].name || \'\').includes(f))\n    .map(c => `<div class="ci${c === activeCode ? \' active\' : \'\'}" onclick="selectCompany(\'${c}\')">\n      <span class="cc">${c}</span>\n      ${companies[c].name ? `<span class="cn">${companies[c].name}</span>` : \'\'}\n    </div>`).join(\'\');\n}\nfunction filterList(v) { renderList(v); }\nrenderList(\'\');\n\nfunction selectCompany(code) {\n  activeCode = code;\n  renderList(document.getElementById(\'srch\').value);\n  document.getElementById(\'ph\').style.display = \'none\';\n  document.getElementById(\'detail\').style.display = \'block\';\n\n  const co = companies[code];\n  document.getElementById(\'ch\').innerHTML =\n    `${code}<small>${co.name || \'\'}</small>`;\n\n  const years = Object.keys(co.data).sort((a, b) => b - a);\n  document.getElementById(\'ybtns\').innerHTML =\n    years.map(y => `<button class="yb" onclick="selectYear(\'${y}\')">${y}</button>`).join(\'\');\n  document.getElementById(\'qs\').innerHTML = \'\';\n\n  if (years.length === 1) selectYear(years[0]);\n}\n\nconst PERIOD_ORDER = [\'Q1財報\',\'Q2財報\',\'Q3財報\',\'Q4財報\',\'年報(大型股/金融)\',\'年報(其餘)\',\'月營收\',\'其他\'];\n\nfunction selectYear(yr) {\n  document.querySelectorAll(\'.yb\').forEach(b =>\n    b.classList.toggle(\'active\', b.textContent === yr));\n\n  const data = companies[activeCode].data[yr] || {};\n  const qs = document.getElementById(\'qs\');\n  qs.innerHTML = \'\';\n\n  PERIOD_ORDER.forEach(period => {\n    if (!data[period]) return;\n    const { before, after, same } = data[period];\n    if (!before.length && !after.length && !same.length) return;\n\n    const sB = [...before].sort((a, b) => a[\'距截止日交易日\'] - b[\'距截止日交易日\']);\n    const sA = [...after].sort((a, b) => a[\'距截止日交易日\'] - b[\'距截止日交易日\']);\n\n    function tbl(arr, cls) {\n      if (!arr.length) return \'<div class="empty">無資料</div>\';\n      return `<table>\n        <tr><th>日期</th><th>時間</th><th>交易日差</th><th>財報截止日</th><th>市場</th></tr>\n        ${arr.map(r => `<tr>\n          <td>${r[\'日期\'] || \'\'}</td>\n          <td>${r[\'召開法人說明會時間\'] || \'\'}</td>\n          <td class="${cls}">${r[\'距截止日交易日\']}</td>\n          <td>${r[\'財報錨點\'] || \'\'}</td>\n          <td>${r[\'市場\'] || \'\'}</td>\n        </tr>`).join(\'\')}\n      </table>`;\n    }\n\n    const div = document.createElement(\'div\');\n    div.className = \'qc\';\n    div.innerHTML = `\n      <div class="qh">${period}</div>\n      <div class="qbody">\n        <div class="qcol">\n          <div class="qst st-b">截止日前（${before.length} 筆）</div>\n          ${tbl(sB, \'dn\')}\n        </div>\n        <div class="qcol">\n          <div class="qst st-a">截止日後（${after.length} 筆）</div>\n          ${tbl(sA, \'dp\')}\n        </div>\n        ${same.length ? `<div class="qcol">\n          <div class="qst st-s">截止日當天（${same.length} 筆）</div>\n          ${tbl(same, \'dz\')}\n        </div>` : \'\'}\n      </div>`;\n    qs.appendChild(div);\n  });\n}\n</script>\n</body>\n</html>'

_html_out = _template.replace('__DATA__', _data_json)
_out_path = r'E:\法說會+主動型\法說會查詢.html'
with open(_out_path, 'w', encoding='utf-8') as _f:
    _f.write(_html_out)
print(f'已輸出：{_out_path}')
print(f'共 {len(_ex):,} 筆資料已嵌入，請用瀏覽器直接開啟 法說會查詢.html')
